# Stage 12 — Mars Threshold and Prediction Analysis

| Field | Value |
|---|---|
| **Pipeline stage** | Stage 12 — Mars threshold and prediction analysis |
| **Previous stage** | Stage 11 — Mars inference (`notebooks/regime/01_mars_inference.ipynb`) |
| **Next stage** | Stage 13 — Scientific interpretation (`notebooks/interpretation/00_scientific_summary.ipynb`) |
| **Purpose** | Analyse how Mars prediction interpretation changes under different threshold choices. Sweeps operating threshold from 0.1 to 0.9, shows touching fraction and network-level statistics, and compares results across regA/B/C. Earth-calibrated thresholds are shown as reference lines. |
| **Inputs** | `data/Mars/model_outputs/mars_combined_reg{A,B,C}_predictions.parquet`; `models/optimal_threshold_geom_plus_cnn_emb_reg{A,B,C}.txt` |
| **Outputs** | Threshold comparison plots; optional `data/Mars/model_outputs/threshold_sensitivity_summary.csv` |
| **Decision gate** | Choose operating threshold for scientific interpretation. Inform Stage 13. |

## 0. Configuration

In [ ]:
# Thresholds to sweep
SWEEP = [0.10, 0.20, 0.30, 0.40, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.90]

# Write threshold sensitivity CSV to data/Mars/model_outputs/
WRITE_CSV = True

# Minimum network pairs to include in network-level stats.
MIN_PAIRS_PER_NETWORK = 3

## 1. Imports and load predictions

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from channel_heads.io.paths import MARS_MODEL_OUTPUTS_DIR, MODELS_DIR

REGIMES_ORDER = ['regA', 'regB', 'regC']
COLORS = {'regA': '#e41a1c', 'regB': '#377eb8', 'regC': '#4daf4a'}

preds: dict[str, pd.DataFrame] = {}
thresholds: dict[str, float] = {}

for reg in REGIMES_ORDER:
    pred_path = MARS_MODEL_OUTPUTS_DIR / f"mars_combined_{reg}_predictions.parquet"
    thr_path  = MODELS_DIR / f"optimal_threshold_geom_plus_cnn_emb_{reg}.txt"

    if not pred_path.exists():
        print(f"WARNING: {pred_path} not found — skipping {reg}")
        continue

    preds[reg] = pd.read_parquet(pred_path)
    if thr_path.exists():
        thresholds[reg] = float(thr_path.read_text().strip())
    print(f"{reg}: {len(preds[reg]):,} pairs  Earth threshold={thresholds.get(reg, 'N/A')}")

if not preds:
    raise RuntimeError("No prediction files found. Run Stage 11 first.")

## 2. Pair-level probability distributions

In [ ]:
fig, axes = plt.subplots(1, len(preds), figsize=(5 * len(preds), 4), sharey=True)
if len(preds) == 1:
    axes = [axes]

for ax, (reg, df) in zip(axes, preds.items()):
    ax.hist(df['prob_touching'], bins=50, color=COLORS[reg], edgecolor='white', alpha=0.85)
    if reg in thresholds:
        ax.axvline(thresholds[reg], color='black', linestyle='--', linewidth=1.5,
                   label=f'Earth thr = {thresholds[reg]:.3f}')
        ax.legend(fontsize=8)
    ax.set_xlabel('P(touching)')
    ax.set_ylabel('Pair count' if ax == axes[0] else '')
    ax.set_title(f'{reg}  (n={len(df):,})')

fig.suptitle('Mars prediction probability distribution by regime', fontsize=11)
fig.tight_layout()
plt.show()

## 3. Touching fraction vs threshold sweep

In [ ]:
rows = []
for reg, df in preds.items():
    for thr in SWEEP:
        predicted = (df['prob_touching'] >= thr).sum()
        rows.append({
            'regime': reg,
            'threshold': thr,
            'n_touching': int(predicted),
            'n_total': len(df),
            'touching_fraction': predicted / len(df),
        })

sweep_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(9, 5))
for reg in preds:
    sub = sweep_df[sweep_df['regime'] == reg]
    ax.plot(sub['threshold'], sub['touching_fraction'],
            'o-', color=COLORS[reg], label=reg, linewidth=2, markersize=5)
    if reg in thresholds:
        ax.axvline(thresholds[reg], color=COLORS[reg], linestyle=':', linewidth=1)

ax.set_xlabel('Operating threshold')
ax.set_ylabel('Fraction of pairs predicted touching')
ax.set_title('Mars touching fraction vs operating threshold')
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
ax.set_ylim(0, 1)
fig.tight_layout()
plt.show()

## 4. High-confidence pair counts

In [ ]:
HIGH_CONF_THRESHOLDS = [0.80, 0.85, 0.90, 0.95]

print(f"{'Regime':<8}  {'Threshold':>10}  {'n_high_conf':>12}  {'pct_total':>10}")
print('-' * 46)
for reg, df in preds.items():
    for thr in HIGH_CONF_THRESHOLDS:
        n = int((df['prob_touching'] >= thr).sum())
        pct = 100 * n / len(df)
        print(f"{reg:<8}  {thr:>10.2f}  {n:>12,}  {pct:>9.1f}%")
    print()

## 5. Network-level statistics at Earth-calibrated threshold

In [ ]:
net_stats_all = []

for reg, df in preds.items():
    thr = thresholds.get(reg, 0.5)
    df2 = df.copy()
    df2['pred'] = (df2['prob_touching'] >= thr).astype(int)

    net_grp = df2.groupby('network_id').agg(
        n_pairs=('pred', 'count'),
        n_touching=('pred', 'sum'),
        mean_prob=('prob_touching', 'mean'),
    ).reset_index()
    net_grp = net_grp[net_grp['n_pairs'] >= MIN_PAIRS_PER_NETWORK]
    net_grp['touching_frac'] = net_grp['n_touching'] / net_grp['n_pairs']
    net_grp['regime'] = reg
    net_stats_all.append(net_grp)

    print(f"{reg} (thr={thr:.3f}): {len(net_grp)} networks with ≥{MIN_PAIRS_PER_NETWORK} pairs")
    print(f"  Networks with any touching  : {(net_grp['n_touching'] > 0).sum()}")
    print(f"  Mean touching frac          : {net_grp['touching_frac'].mean():.3f}")
    print()

net_stats = pd.concat(net_stats_all, ignore_index=True) if net_stats_all else pd.DataFrame()

In [ ]:
if not net_stats.empty and 'touching_frac' in net_stats.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    for reg in preds:
        sub = net_stats[net_stats['regime'] == reg]
        ax.hist(sub['touching_frac'], bins=20, alpha=0.55,
                color=COLORS[reg], label=reg, density=True)
    ax.set_xlabel('Touching fraction per network')
    ax.set_ylabel('Density')
    ax.set_title('Network-level touching fraction (at Earth-calibrated threshold)')
    ax.legend()
    fig.tight_layout()
    plt.show()

## 6. Regime comparison at Earth-calibrated thresholds

In [ ]:
compare_rows = []
for reg, df in preds.items():
    thr = thresholds.get(reg, 0.5)
    n_total = len(df)
    n_touching = int((df['prob_touching'] >= thr).sum())
    compare_rows.append({
        'regime': reg,
        'earth_threshold': round(thr, 4),
        'n_pairs': n_total,
        'n_touching': n_touching,
        'touching_pct': round(100 * n_touching / n_total, 2),
        'n_networks': preds[reg]['network_id'].nunique(),
    })

compare_df = pd.DataFrame(compare_rows)
print("Regime comparison at Earth-calibrated thresholds:")
display(compare_df)

## 7. Write threshold sensitivity summary (optional)

In [ ]:
if WRITE_CSV:
    out_path = MARS_MODEL_OUTPUTS_DIR / "threshold_sensitivity_summary.csv"
    sweep_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

    compare_out = MARS_MODEL_OUTPUTS_DIR / "regime_comparison_at_earth_threshold.csv"
    compare_df.to_csv(compare_out, index=False)
    print(f"Saved: {compare_out}")
else:
    print("Set WRITE_CSV = True to save summaries.")